## Summarize the contents of `species-genes.csv`

In [55]:
import polars as pl

In [56]:
sg_df = pl.read_csv('../outputs.cds/singleclust/species-genes.csv')

In [57]:
good_df = sg_df.filter(pl.col("good") != 0)
anchor_df = sg_df.filter(pl.col("anchor") == 1)

In [58]:
good_df['species'].value_counts().sort(by='count', descending=True)

species,count
str,u32
"""s__Mogibacterium_A kristiansen…",9
"""s__Cryptobacteroides sp0004326…",6
"""s__Ornithospirochaeta sp022785…",6
"""s__Cryptobacteroides sp0340892…",6
"""s__Prevotella sp002251295""",5
…,…
"""s__Lactobacillus amylovorus""",3
"""s__Bariatricus sp004560705""",3
"""s__UBA2868 sp004552595""",3


In [59]:
names_df = []
for line in open('../inputs.cds/names.list'):
    name = line.strip()
    names_df.append(dict(species=name))
names_df = pl.DataFrame(names_df)

In [60]:
join_df = good_df.join(names_df, on='species', how='full', coalesce=True)
join_df

good,anchor,species,gene_name,description
i64,i64,str,str,str
1,1,"""s__Phascolarctobacterium_A suc…","""CNENGHLA_01260""","""BLAST match to hydrogenase lar…"
1,0,"""s__Phascolarctobacterium_A suc…","""EHOAPHDI_01174""","""BLAST match to protein phospha…"
1,0,"""s__Phascolarctobacterium_A suc…","""CNENGHLA_00658""","""BLAST match to 4-hydroxy-3-met…"
1,0,"""s__Phascolarctobacterium_A suc…","""IFIBFMPA_00800""","""BLAST match to 2-isopropylmal…"
1,0,"""s__Lactobacillus amylovorus""","""BBOFCOCJ_01349""","""BLAST match to peptidase T [La…"
…,…,…,…,…
1,0,"""s__Cryptobacteroides sp0004349…","""KHMCBGLI_01458""","""putative uncharacterized prote…"
1,0,"""s__Cryptobacteroides sp0004349…","""FCHBMNJF_01297""","""ATP-binding protein [Bacteroid…"
null,null,"""s__Fimisoma sp002320005""",null,null


In [61]:
anchor_count_df = (join_df.group_by('species')
                   .agg(anchor_count=pl.col('anchor').sum(),
                        good_count=pl.col('good').sum())
                   .sort(by='anchor_count'))

with pl.Config(tbl_rows=-1):
    print(anchor_count_df)

shape: (19, 3)
┌─────────────────────────────────┬──────────────┬────────────┐
│ species                         ┆ anchor_count ┆ good_count │
│ ---                             ┆ ---          ┆ ---        │
│ str                             ┆ i64          ┆ i64        │
╞═════════════════════════════════╪══════════════╪════════════╡
│ s__Bariatricus sp004560705      ┆ 0            ┆ 3          │
│ s__Prevotella sp000434975       ┆ 0            ┆ 3          │
│ s__JAFBIX01 sp021531895         ┆ 0            ┆ 0          │
│ s__Fimisoma sp002320005         ┆ 0            ┆ 0          │
│ s__Floccifex porci              ┆ 0            ┆ 0          │
│ s__Holdemanella porci           ┆ 0            ┆ 1          │
│ s__JALFVM01 sp022787145         ┆ 0            ┆ 1          │
│ s__Colivicinus sp002299675      ┆ 1            ┆ 5          │
│ s__Cryptobacteroides sp9005469… ┆ 1            ┆ 4          │
│ s__Mogibacterium_A kristiansen… ┆ 1            ┆ 9          │
│ s__Cryptobacteroides sp